In [37]:
import zipfile
import os
import pandas as pd
from pathlib import Path

import numpy as np, scipy.sparse as sp
import sys 
import torch
from torch.utils.data import Dataset, DataLoader,Subset
import csv
from sklearn.metrics import classification_report, accuracy_score, f1_score
from collections import Counter
import math, time, random
import torch.nn.functional as F

import warnings
warnings.filterwarnings("ignore")

if torch.cuda.is_available(): print("CUDA is available! PyTorch can see the GPU.")
else:print("CUDA is not available. PyTorch will use the CPU.")

os.chdir('/scratch/users/ntu/lizh0106/nscc_work')
print(os.getcwd())

final_filtered_data = pd.read_csv("sampler_context_mha_aggc/AGGC_metadata.csv")
base = "Processed_Features/AGGC" ##这个是文件夹

final_filtered_data.head()

SAVE = True

CUDA is available! PyTorch can see the GPU.
/scratch/users/ntu/lizh0106/nscc_work


In [38]:
PROJECT_ROOT = "/scratch/users/ntu/lizh0106/nscc_work/sampler_context_mha_aggc"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utilities.evaluate_slide_labels import evaluate_oof_classification
from utilities.save import save_json_metrics

In [ ]:
out_dir =  "sampler_context_mha_aggc/l2_2048/cv_results"
os.makedirs(out_dir, exist_ok=True) 

oof_dir =  "sampler_context_mha_aggc/l2_2048/oof_results"
os.makedirs(oof_dir, exist_ok=True)

idx_path = os.path.join(base, "20x_512", "index.csv")
feat_path = os.path.join(base, "20x_512", "features.npy")

master_df = pd.read_csv(idx_path)
master_df ["y"] = master_df ["y"].astype(int)
master_df.head(3)

,slide_id,y,start,length,n_tiles_read,h5_path
0,Subset1_Train_1,1,0,10339,10339,/home/users/ntu/lizh0106/scratch/nscc_work/AGG...
1,Subset1_Train_10,3,10339,14579,14579,/home/users/ntu/lizh0106/scratch/nscc_work/AGG...
2,Subset1_Train_100,3,24918,7562,7562,/home/users/ntu/lizh0106/scratch/nscc_work/AGG...


In [40]:
class PackedDataset(Dataset):
    """
    单尺度 memmap（features.npy）+ 不 padding
    df 需要列：start, length, y, slide_id
    """
    def __init__(self, master_df, base_dir, feature_dim=1024,
                 rel_path=("20x_512", "features.npy"),
                 dtype=np.float32):
        self.df = master_df.reset_index(drop=True)
        self.D = int(feature_dim)
        self.dtype = dtype

        # ✅ memmap 文件路径固定：base_dir/20x_512/features.npy
        self.feat_path = os.path.join(base_dir, *rel_path)
        if not os.path.exists(self.feat_path):
            raise FileNotFoundError(f"Memmap file not found: {self.feat_path}")

        # 懒加载：避免 DataLoader 多进程时 fork 的潜在问题
        self._mm = None

    def _get_mm(self):
        if self._mm is None:
            self._mm = np.memmap(self.feat_path, dtype=self.dtype, mode="r")
            if self._mm.size % self.D != 0:
                raise ValueError(
                    f"Memmap size {self._mm.size} not divisible by D={self.D}. "
                    f"Check feature_dim/dtype for {self.feat_path}"
                )
        return self._mm

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        start = int(row["start"])
        length = int(row["length"])
        y = int(row["y"])
        slide_id = row["slide_id"]

        mm2d = self._get_mm().reshape(-1, self.D)  # 最好挪去缓存，只示意
        X_np = mm2d[start:start + length]

        # 只在必要时拷贝，并保证连续内存（对 torch/from_numpy 更友好）
        X_np = np.ascontiguousarray(X_np)

        X = torch.from_numpy(X_np)  # 已经是 float32 就别 .float()
        return X, torch.tensor(y, dtype=torch.long), slide_id, idx


def collate_varlen(batch):
    Xs, ys, ids,idx = zip(*batch)
    return list(Xs), torch.stack(ys, 0), list(ids),list(idx)



In [41]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ContextMHA(nn.Module):
    def __init__(self, D=1024, H=4, num_classes=5, hidden_dim=512, p_drop=0.2):
        super().__init__()
        self.D = D
        self.H = H
        self.ln_x = nn.LayerNorm(D)

        # each head ONLY ONE w and b
        # We don't transpose here, as good practice is semantically have each row as attention head
        self.attn_w = nn.Parameter(torch.randn(H, D) * (1.0 / (D ** 0.5))) #normalize gradient, due to the attn mechanism
        self.attn_b = nn.Parameter(torch.zeros(H))

        self.classifier = nn.Sequential(
            nn.Linear(D * H, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p_drop),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, Xs, return_attn=False):
        """
        Xs: list of (N_i, D) OR a single tensor (N, D)
        """
        if isinstance(Xs, torch.Tensor):
            Xs = [Xs]

        device = self.attn_w.device
        slide_vecs = []
        attn_list = []  # store A for each slide

        for X in Xs:
            X = X.to(device).float()  # (N, D)
            X = self.ln_x(X)
            scores = torch.sigmoid(X @ self.attn_w.T + self.attn_b)  # (N, H)
            A = F.softmax(scores, dim=0)                             # (N, H)
            Z = A.T @ X                   # (H, D) # So this is a final score for a features that takes  all tiles into consider.

            slide_vecs.append(Z.reshape(-1))                         # (H*D,)
            if return_attn:
                attn_list.append(A.detach().cpu())                   # keep on CPU for saving/visual

        slide_mat = torch.stack(slide_vecs, dim=0)  # (B, H*D)
        logits = self.classifier(slide_mat)

        if return_attn:
            return logits, attn_list   # list length B, each (N_i, H)
        return logits



In [42]:
FEATURE_DIM = 1024

dataset = PackedDataset(master_df, base_dir=base,feature_dim=FEATURE_DIM)

# test batch
subset_idx = list(range(min(8, len(dataset))))
ds_small = Subset(dataset, subset_idx)
loader = DataLoader(ds_small, batch_size=4, shuffle=False, num_workers=2,
                    pin_memory=True, collate_fn=collate_varlen)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = ContextMHA(D=FEATURE_DIM, H=4, num_classes=5, hidden_dim=512).to(device)

Xs, ys, ids,idx = next(iter(loader))
print("Batch lens:", len(Xs), ys.shape, len(ids))  # e.g., 4, torch.Size([4]), 4

with torch.no_grad():
    logits = model(Xs)
print("Logits shape:", logits.shape)              # 应为 (B, 5)


cuda


Batch lens: 4 torch.Size([4]) 4
Logits shape: torch.Size([4, 5])


In [43]:
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def make_class_weights(labels, num_classes):
    cnt = Counter(labels)
    total = sum(cnt.values())
    weights = [total / (num_classes * cnt.get(c, 1)) for c in range(num_classes)]
    return torch.tensor(weights, dtype=torch.float32)

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    for Xs, ys, _ids,idx in loader:
        ys = ys.to(device)
        optimizer.zero_grad()
        logits = model(Xs)
        loss = criterion(logits, ys)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        running_loss += loss.item() * ys.size(0)
    return running_loss / len(loader.dataset)

def evaluate(model, loader, device, num_classes=5, return_attn=False):
    model.eval()

    total_loss = 0.0
    all_y, all_pred = [], []

    
    dummy_weights = torch.ones(num_classes, device=device)
    criterion_eval = nn.CrossEntropyLoss(weight=dummy_weights)

    # 用于 OOF / 可视化的收集器
    all_ids = []
    all_preds = []
    all_probs = []
    attn_dict = {} if return_attn else None

    with torch.no_grad():
        for Xs, ys, _ids, idx in loader:
            ys = ys.to(device)

            out = model(Xs, return_attn=return_attn)
            if return_attn:
                logits, attn_list = out
            else:
                logits = out

            loss = criterion_eval(logits, ys)
            total_loss += loss.item() * ys.size(0)

            probs = F.softmax(logits, dim=1)          # (B, C)
            preds = probs.argmax(dim=1)               # (B,)

            # ===== metrics =====
            all_y.extend(ys.cpu().tolist())
            all_pred.extend(preds.cpu().tolist())

            # ===== OOF / per-sample outputs =====
            all_ids.extend(idx)
            all_preds.extend(preds.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

            if return_attn:
                for sid, A in zip(_ids, attn_list):
                    attn_dict[sid] = A  # A 已经是 cpu tensor（或你可以在这 detach）

    # ===== stack to numpy =====
    all_preds = np.asarray(all_preds)                 # (N,)
    all_probs = np.concatenate(all_probs, axis=0)     # (N, C)

    acc = accuracy_score(all_y, all_pred)
    macro_f1 = f1_score(all_y, all_pred, average="macro")
    report = classification_report(all_y, all_pred, digits=3)

    return (
        total_loss / len(loader.dataset),
        acc,
        macro_f1,
        report,
        all_ids,        # list[int] or list[str]
        all_preds,      # np.ndarray (N,)
        all_probs,      # np.ndarray (N, C)
        attn_dict,      # dict[id -> (N_i, H)] or None
    )




In [44]:
D = 1024
H = 4

labels = master_df["y"].tolist()
num_classes = len(set(labels))  ###5

from sklearn.model_selection import StratifiedKFold

set_seed(42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [45]:
print("len labels: ",len(labels))

len labels:  187


In [46]:
model_tmp = ContextMHA(D=1024,H=H,num_classes=num_classes,hidden_dim=512,p_drop=0.2).to(device)

print(model_tmp)

# 参数统计
total_params = sum(p.numel() for p in model_tmp.parameters())
trainable_params = sum(p.numel() for p in model_tmp.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

del model_tmp

ContextMHA(
  (ln_x): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  (classifier): Sequential(
    (0): Linear(in_features=4096, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=512, out_features=5, bias=True)
  )
)

Total parameters: 2,106,377
Trainable parameters: 2,106,377


In [47]:
fold_summaries = [] 
N ,C= len(labels),num_classes

# Store out-of-fold predicted probabilities for each sample (N, C)
oof_pred = np.zeros(len(master_df), dtype=np.int64)
oof_prob = np.zeros((len(master_df), num_classes), dtype=np.float32)
all_attn = {}

for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(labels)), labels), 1):
    print(f"\n===== Fold {fold} / {skf.n_splits} =====")
    ds_train = Subset(dataset, tr_idx)
    ds_valid = Subset(dataset, va_idx)

    train_loader = DataLoader(ds_train, batch_size=8, shuffle=True, num_workers=4,
                              collate_fn=collate_varlen, pin_memory=True)
    valid_loader = DataLoader(ds_valid, batch_size=8, shuffle=False, num_workers=4,
                              collate_fn=collate_varlen, pin_memory=True)

    model = ContextMHA(D=D, H=H, num_classes=num_classes,
                       hidden_dim=512, p_drop=0.2).to(device)

    # 类别权重
    y_train_fold = [labels[i] for i in tr_idx]
    class_weights = make_class_weights(y_train_fold, num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max',
                                                           factor=0.5, patience=2, verbose=True)

    # 日志与早停
    best_f1, best_state = -1.0, None
    best_epoch = 0
    patience, bad = 5, 0
    epochs = 60
    log_rows = []

    for epoch in range(1, epochs + 1):
        print("EPOCH : ", epoch)
        t0 = time.time()
        tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        va_loss, va_acc, va_f1, va_rep, _,_,_,_ = evaluate(model, valid_loader, device, num_classes=num_classes)
        scheduler.step(va_f1)

        elapsed = time.time() - t0
        print(f"Epoch {epoch:02d}  train_loss={tr_loss:.4f}  val_loss={va_loss:.4f}  "
              f"val_acc={va_acc:.4f}  val_macroF1={va_f1:.4f}  ({elapsed:.1f}s)")

        # 记录 epoch 日志
        log_rows.append({
            "fold": fold,
            "epoch": epoch,
            "train_loss": tr_loss,
            "val_loss": va_loss,
            "val_acc": va_acc,
            "val_f1": va_f1,
            "time_sec": elapsed,
        })

        # early stopping
        if va_f1 > best_f1 + 1e-4:
            best_f1 = va_f1
            best_epoch = epoch
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print(f"Early stopping at epoch {epoch}. Best epoch = {best_epoch}")
                break

    # 保存每折日志 CSV
    log_path = os.path.join(out_dir, f"fold{fold}_log.csv")
    with open(log_path, "w", newline="") as fw:
        writer = csv.DictWriter(fw, fieldnames=log_rows[0].keys())
        writer.writeheader()
        writer.writerows(log_rows)

    # 保存最佳模型
    model_path = os.path.join(out_dir, f"fold{fold}_best.pt")
    if best_state is not None:
        torch.save(best_state, model_path)
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    va_loss, va_acc, va_f1, va_rep, ids, preds, probs, attn_dict = evaluate(model, valid_loader, device, num_classes=num_classes, return_attn=True)

    oof_pred[ids] = preds
    oof_prob[ids] = probs

    all_attn.update(attn_dict)

    # 汇总当前折指标
    fold_summaries.append({
        "fold": fold,
        "best_epoch": best_epoch,
        "val_acc": va_acc,
        "val_f1": va_f1,
        "val_loss": va_loss,
        "best_model_path": model_path,
        "log_path": log_path
    })

# === 最终汇总 ===
df_summary = pd.DataFrame(fold_summaries)
df_summary.to_csv(os.path.join(out_dir, "cv_summary.csv"), index=False)

print("\n==== 5-Fold Summary ====")
print(df_summary[["fold","val_acc","val_f1","best_epoch"]])
print(f"\nMean F1={df_summary['val_f1'].mean():.4f},  Std={df_summary['val_f1'].std():.4f}")




===== Fold 1 / 5 =====
EPOCH :  1


Epoch 01  train_loss=2.4439  val_loss=2.4264  val_acc=0.1842  val_macroF1=0.1392  (5.2s)
EPOCH :  2
Epoch 02  train_loss=1.5833  val_loss=1.3139  val_acc=0.4211  val_macroF1=0.3195  (2.9s)
EPOCH :  3
Epoch 03  train_loss=1.5894  val_loss=3.4088  val_acc=0.0789  val_macroF1=0.1111  (2.8s)
EPOCH :  4
Epoch 04  train_loss=1.7940  val_loss=1.5900  val_acc=0.5000  val_macroF1=0.3016  (2.8s)
EPOCH :  5
Epoch 05  train_loss=1.2770  val_loss=1.2261  val_acc=0.5000  val_macroF1=0.2542  (2.8s)
EPOCH :  6
Epoch 06  train_loss=0.8619  val_loss=1.1049  val_acc=0.5526  val_macroF1=0.3857  (2.8s)
EPOCH :  7
Epoch 07  train_loss=1.1176  val_loss=1.4771  val_acc=0.3947  val_macroF1=0.3800  (2.8s)
EPOCH :  8
Epoch 08  train_loss=0.7676  val_loss=1.1519  val_acc=0.6053  val_macroF1=0.4266  (2.8s)
EPOCH :  9
Epoch 09  train_loss=0.7446  val_loss=1.2136  val_acc=0.5789  val_macroF1=0.4185  (2.8s)
EPOCH :  10
Epoch 10  train_loss=0.7209  val_loss=1.3068  val_acc=0.6316  val_macroF1=0.5604  (2.8s)
EPOCH :  1

In [48]:
attn_np_dict = {
    k: v.detach().cpu().numpy()
    for k, v in all_attn.items()
}

In [49]:
if SAVE:

    np.save(os.path.join(oof_dir, "aggc_oof_tile_labels.npy"), oof_pred)
    np.save(os.path.join(oof_dir, "aggc_oof_slide_proba.npy"), oof_prob)
    np.savez(os.path.join(oof_dir, "attn_map_aggc.npz"), **attn_np_dict)

In [50]:
oof_pred = np.load(os.path.join(oof_dir, "aggc_oof_tile_labels.npy"))
oof_prob = np.load(os.path.join(oof_dir, "aggc_oof_slide_proba.npy"))

In [51]:
aggc_oof_result = evaluate_oof_classification(master_df["y"].values,oof_pred,oof_prob,labels=[0,1,2,3,4],verbose = True,)
save_json_metrics(aggc_oof_result, os.path.join(oof_dir,"metrics_aggc.json"))

=== OOF Evaluation ===
Labels: [0, 1, 2, 3, 4]
Accuracy: 0.5989304812834224
Balanced accuracy: 0.5538461538461539
Confusion matrix:
 [[ 6  5  0  0  0]
 [12 60  2  4  0]
 [ 4 23 30  6  3]
 [ 2  0  1  5  2]
 [ 1  3  4  3 11]]
Classification report:
               precision    recall  f1-score   support

           0       0.24      0.55      0.33        11
           1       0.66      0.77      0.71        78
           2       0.81      0.45      0.58        66
           3       0.28      0.50      0.36        10
           4       0.69      0.50      0.58        22

    accuracy                           0.60       187
   macro avg       0.54      0.55      0.51       187
weighted avg       0.67      0.60      0.61       187

AUC per class: {0: 0.7417355371900826, 1: 0.7675840978593272, 2: 0.7176308539944903, 3: 0.7022598870056498, 4: 0.850137741046832}
Macro AUC: 0.7558696234192764


In [52]:
# =========================
# TCGA inference + 5-fold ensemble
# =========================

# 1) 路径与 index 读入（修正你最后一行的 bug）
TCGA_base_dir = "Processed_Features/TCGA_PRAD/tcga_without_anno_arrays"
TCGA_features = os.path.join(TCGA_base_dir, "20x_512", "features.npy")
TCGA_f_index  = os.path.join(TCGA_base_dir, "20x_512", "index.csv")

df_tcga = pd.read_csv(TCGA_f_index)   # ✅ 用变量，不要加引号
if "y" not in df_tcga.columns:
    raise ValueError("df_tcga must have column 'y' (same format as AGGC).")
df_tcga["y"] = df_tcga["y"].astype(int)

# 2) dataset / loader（沿用你写好的 PackedDataset + collate_varlen）
tcga_dataset = PackedDataset(df_tcga, base_dir=TCGA_base_dir, feature_dim=FEATURE_DIM)

# 推理推荐 batch 稍微大点；但你这个模型 forward 是 list-of-tensors，太大也会慢
tcga_loader = DataLoader(
    tcga_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    collate_fn=collate_varlen
)

# 3) 找到你保存的 5 个 fold 模型路径（就是你训练时保存的 fold{fold}_best.pt）
# out_dir = "sampler_context_mha_aggc/no_l2/cv_results"  # 你上面已经定义过
model_paths = [os.path.join(out_dir, f"fold{f}_best.pt") for f in range(1, 6)]
missing = [p for p in model_paths if not os.path.exists(p)]
if missing: raise FileNotFoundError(f"Missing model checkpoints:\n" + "\n".join(missing))

# 4) 逐模型跑 evaluate，收集每个样本的 probs（按 idx 回填，避免顺序风险）
N_tcga = len(tcga_dataset)
C = num_classes  # 你上面训练时算过（应该是 5）
ensemble_prob = np.zeros((N_tcga, C), dtype=np.float32)

per_model_reports = []

for mi, ckpt_path in enumerate(model_paths, 1):
    print(f"\n[TCGA] Running model {mi}/5: {ckpt_path}")

    m = ContextMHA(D=D, H=H, num_classes=C, hidden_dim=512, p_drop=0.2).to(device)
    state = torch.load(ckpt_path, map_location="cpu")
    m.load_state_dict(state, strict=True)
    m.to(device)

    # 这里你的 evaluate 会返回：loss, acc, f1, report, ids, preds, probs, attn_dict
    tcga_loss, tcga_acc, tcga_f1, tcga_rep, ids, preds, probs, _ = evaluate(
        m, tcga_loader, device, num_classes=C, return_attn=False
    )

    # 回填到 (N, C)；ids 是 dataset index
    ids = np.asarray(ids, dtype=np.int64)
    ensemble_prob[ids] += probs.astype(np.float32)

    per_model_reports.append({
        "model_idx": mi,
        "ckpt": ckpt_path,
        "loss": float(tcga_loss),
        "acc": float(tcga_acc),
        "macro_f1": float(tcga_f1),
    })

# 5) ensemble average + hard pred
ensemble_prob /= len(model_paths)
tcga_pred = ensemble_prob.argmax(axis=1).astype(np.int64)

# 6) 调用你已有的 evaluate_oof_classification（df_tcga 的格式说和 aggc 一样，所以直接套）
tcga_result = evaluate_oof_classification(
    df_tcga["y"].values,
    tcga_pred,
    ensemble_prob,
    labels=[0, 1, 2, 3, 4],
    verbose=True,
)

# 7) 保存（可选）
tcga_out_dir = os.path.join(oof_dir, "tcga_eval")
os.makedirs(tcga_out_dir, exist_ok=True)

np.save(os.path.join(tcga_out_dir, "tcga_ensemble_pred.npy"), tcga_pred)
np.save(os.path.join(tcga_out_dir, "tcga_ensemble_prob.npy"), ensemble_prob)
save_json_metrics(tcga_result, os.path.join(tcga_out_dir, "metrics_tcga_ensemble.json"))

# 额外存每个模型自己的简表（方便你看有没有某个fold特别拉胯）
pd.DataFrame(per_model_reports).to_csv(
    os.path.join(tcga_out_dir, "per_model_summary.csv"),
    index=False
)

print("\n[TCGA] Done. Saved to:", tcga_out_dir)



[TCGA] Running model 1/5: sampler_context_mha_aggc/l2/cv_results/fold1_best.pt



[TCGA] Running model 2/5: sampler_context_mha_aggc/l2/cv_results/fold2_best.pt

[TCGA] Running model 3/5: sampler_context_mha_aggc/l2/cv_results/fold3_best.pt

[TCGA] Running model 4/5: sampler_context_mha_aggc/l2/cv_results/fold4_best.pt

[TCGA] Running model 5/5: sampler_context_mha_aggc/l2/cv_results/fold5_best.pt
=== OOF Evaluation ===
Labels: [0, 1, 2, 3, 4]
Accuracy: 0.439873417721519
Balanced accuracy: 0.3669914528410918
Confusion matrix:
 [[ 5 18  2  0  0]
 [ 8 60 21  9  0]
 [ 5 18 44  4  2]
 [ 4  6 18  4  2]
 [ 0  7 40 13 26]]
Classification report:
               precision    recall  f1-score   support

           0       0.23      0.20      0.21        25
           1       0.55      0.61      0.58        98
           2       0.35      0.60      0.44        73
           3       0.13      0.12      0.12        34
           4       0.87      0.30      0.45        86

    accuracy                           0.44       316
   macro avg       0.43      0.37      0.36       316

- Find a corrected classified sample
- Define a method to extract thumbnail
- Define 2 methods to visualiza map on image  